# Predicting Data Science Job Salaries: Exploratory Data Analysis and Model Comparison

This notebook explores a dataset of data science job characteristics (experience level, job title, company size, location, etc.) to understand patterns in salaries.

The predictive task is to **predict data science job salaries based on these characteristics**.

Several models (Linear Regression, Ridge Regression, Random Forest) are tested for exploratory comparison.
Results are exploratory and not used for the final model evaluation.

## Data Upload and Extraction

The dataset was downloaded from Kaggle as a ZIP file and uploaded to Google Colab. The ZIP file is extracted to access the CSV data for analysis.


In [1]:
# Upload dataset ZIP file to Google Colab
from google.colab import files
files.upload()
# upload the ZIP
!unzip archive.zip

Saving archive.zip to archive.zip
Archive:  archive.zip
  inflating: Data Science Jobs Salaries.csv  


In [2]:
# List extracted files
!ls

 archive.zip  'Data Science Jobs Salaries.csv'	 sample_data


## Importing Required Libraries

This section imports the Python libraries used for data manipulation, preprocessing, modeling, and evaluation throughout the analysis.

In [3]:
# Libraries for data handling
import pandas as pd
import numpy as np

# Libraries for model training and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Regression models
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

# Evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Data Loading

This section checks the first five rows to ensure that there is data.

In [4]:
df = pd.read_csv("Data Science Jobs Salaries.csv")
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2021e,EN,FT,Data Science Consultant,54000,EUR,64369,DE,50,DE,L
1,2020,SE,FT,Data Scientist,60000,EUR,68428,GR,100,US,L
2,2021e,EX,FT,Head of Data Science,85000,USD,85000,RU,0,RU,M
3,2021e,EX,FT,Head of Data,230000,USD,230000,RU,50,RU,L
4,2021e,EN,FT,Machine Learning Engineer,125000,USD,125000,US,100,US,S


## Feature Selection

The target variable for this analysis is annual salary in USD. Predictor variables include job and company characteristics that are expected to influence compensation, such as **experience level, employment type, job title, remote work ratio, company size, and company location**.

In [5]:
# Define target variable
y = df["salary_in_usd"]

# Define feature matrix
X = df[
    [
        "experience_level",
        "employment_type",
        "job_title",
        "remote_ratio",
        "company_size",
        "company_location"
    ]
]

# Check dimensions
X.shape, y.shape

((245, 6), (245,))

## Data Preprocessing

Categorical features are one-hot encoded so that the models can interpret them numerically,
**while numeric features are left unchanged**.
This preprocessing is wrapped in a ColumnTransformer, which ensures consistent transformations
when used in model pipelines.

In [6]:
# Define categorical and numeric features
categorical_features = [
    "experience_level",
    "employment_type",
    "job_title",
    "company_size",
    "company_location"
]

numeric_features = ["remote_ratio"]

# Set up preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

## Exploratory Model Comparison (Train-Test Split)

To get a rough sense of model performance, we temporarily split the dataset into training and testing sets.
This split is **for exploration only** and will not be used for the final model evaluation.

In [7]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Exploratory Model: Linear Regression

Linear Regression is used as a baseline model because it is simple and interpretable.

It assumes a linear relationship between job features (experience level, job title, location, etc.) and salary.
This model helps provide a benchmark for comparing more complex models.

In [8]:
lr_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

# Calculate RMSE by taking the square root of MSE
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

rmse_lr, mae_lr, r2_lr

(np.float64(83934.12866449943), 43077.01897556139, -0.08461130894010371)

## Exploratory Model: Ridge Regression

Ridge Regression is a regularized version of Linear Regression.

It adds an L2 penalty on the size of coefficients to reduce overfitting, particularly useful for datasets with many one-hot encoded categorical features.
We include it here to see if regularization improves performance over the baseline linear model.

In [9]:
ridge_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0, random_state=42))
    ]
)

ridge_model.fit(X_train, y_train)

y_pred_ridge = ridge_model.predict(X_test)

rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

rmse_ridge, mae_ridge, r2_ridge

(np.float64(68909.7158050195), 40042.953260801725, 0.268931833580474)

## Exploratory Model: Lasso Regression

Lasso Regression is explored as an alternative linear model that applies L1 regularization.

Unlike Ridge Regression, Lasso can shrink some coefficients exactly to zero, effectively performing feature selection.
This model is included for exploratory comparison and is not used as the final model.

In [10]:
lasso_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Lasso(alpha=0.1, random_state=42, max_iter=2000)) # Increased max_iter for convergence
    ]
)

lasso_model.fit(X_train, y_train)

y_pred_lasso = lasso_model.predict(X_test)

rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

rmse_lasso, mae_lasso, r2_lasso

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:656: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 55720174360.5086, tolerance: 140083203.77365306
  model = cd_fast.sparse_enet_coordinate_descent(


(np.float64(84602.93575524146), 44116.97349235297, -0.10196505546811352)

## Exploratory Model: Random Forest Regressor

Random Forest is a non-linear ensemble model that builds multiple decision trees and averages their predictions.
It can capture interactions between features that linear models cannot.

We test it here to see whether a more flexible model improves predictive performance.

In [11]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

rmse_rf, mae_rf, r2_rf

(np.float64(69990.41767354462), 39327.44102526725, 0.24582153687727049)

## Exploratory Model Comparison

Exploratory models are compared using RMSE, MAE, and R² to determine which model provides the best predictive performance.

In [12]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Ridge Regression", "Lasso Regression"],
    "RMSE": [rmse_lr, rmse_rf, rmse_ridge, rmse_lasso],
    "MAE": [mae_lr, mae_rf, mae_ridge, mae_lasso],
    "R2": [r2_lr, r2_rf, r2_ridge, r2_lasso]
})

results

,Model,RMSE,MAE,R2
0,Linear Regression,83934.128664,43077.018976,-0.084611
1,Random Forest,69990.417674,39327.441025,0.245822
2,Ridge Regression,68909.715805,40042.953261,0.268932
3,Lasso Regression,84602.935755,44116.973492,-0.101965


### Analysis

- **Linear Regression** and **Lasso Regression** perform poorly with a negative R² (-0.08) and (-0.10) respectively, indicating they do not capture the variation in salaries well.  
- **Random Forest** shows improvement over linear regression, with a positive R² (0.25) and lower RMSE/MAE.  
- **Ridge Regression** performs the best among the four, with the lowest RMSE (68,910), slightly higher MAE than Random Forest, and the highest R² (0.27).  

Based on these metrics, **Ridge Regression appears to offer the best balance of performance and interpretability** for this dataset.
The modeling notebook will focus on building and evaluating Ridge Regression as the final model.